In [1]:
import pandas as pd
import numpy as np
import re

#orginele dataset inlezen
df = pd.read_csv('Leesmateriaal.csv')
df = df.convert_dtypes()

In [2]:
#Analyse van de kolommen Themas_boek, trefwoorden en bron_url
def normaliseer_waarde(waarde):
    if pd.isna(waarde):
        return ""

    waarde = str(waarde).strip().casefold()

    # Spaties rondom puntkomma's negeren
    waarde = re.sub(r"\s*;\s*", ";", waarde)

    # Meerdere spaties vervangen door één spatie
    waarde = re.sub(r"\s+", " ", waarde)

    return waarde

# Waarden normaliseren
themas = df["Themas_boek"].map(normaliseer_waarde)
trefwoorden = df["trefwoorden"].map(normaliseer_waarde)
bron_url = df["bron_url"].map(normaliseer_waarde)

# Eerst bepalen of Themas_boek een URL bevat
is_url = themas.str.startswith(("http://", "https://"))

# Als Themas_boek een URL is, vergelijk met bron_url.
# Anders vergelijk met trefwoorden.
verwachte_waarde = trefwoorden.copy()
verwachte_waarde.loc[is_url] = bron_url.loc[is_url]

# Controleren welke rijen niet overeenkomen
verschilt = themas != verwachte_waarde

# Alleen afwijkende rijen weergeven
verschillen = df.loc[
    verschilt,
    [
        "Titel",
        "Themas_boek",
        "trefwoorden",
        "bron_url",
    ],
].copy()

# Laat zien waarmee Themas_boek per rij is vergeleken
verschillen["vergeleken_met"] = is_url.loc[verschillen.index].map(
    {
        True: "bron_url",
        False: "trefwoorden",
    }
)

if verschillen.empty:
    print("Alle waarden komen overeen.")
else:
    display(verschillen)

,Titel,Themas_boek,trefwoorden,bron_url,vergeleken_met
188,Zolang het tuin duurt 2,"zelfreflectie,doorzetten",zelfreflectie; doorzetten,<NA>,trefwoorden
201,De kleine kust,"jongeren,vriendschap",jongeren; vriendschap,<NA>,trefwoorden
231,Wat Joris niet wist,"doorzetten,liefde",doorzetten; liefde,<NA>,trefwoorden
242,Sleutel zonder kust,liefde/samenleving,liefde; samenleving,<NA>,trefwoorden
246,De brief van Jelle,geheimen|macht,geheimen; macht,<NA>,trefwoorden
254,Een foto voor Sem,"veerkracht,oorlog",veerkracht; oorlog,<NA>,trefwoorden
270,Het witte huis,reizen/oorlog,reizen; oorlog,<NA>,trefwoorden
282,Grens in de haven,"doorzetten,stalking",doorzetten; stalking,<NA>,trefwoorden


In [3]:
#Analyse van de kolommen genre en genre_kopie

# Waarden uit beide kolommen ophalen en normaliseren
genre = df["genre"].map(normaliseer_waarde)
genre_kopie = df["genre_kopie"].map(normaliseer_waarde)

# Alleen vergelijken als genre_kopie oorspronkelijk geen NaN is
heeft_genre_kopie = df["genre_kopie"].notna()

# Verschillen bepalen
verschilt_genre = (
    heeft_genre_kopie
    & (genre != genre_kopie)
)

# Alleen afwijkende rijen weergeven
genre_verschillen = df.loc[
    verschilt_genre,
    [
        "Titel",
        "genre",
        "genre_kopie",
    ],
].copy()

if genre_verschillen.empty:
    print("Alle ingevulde waarden in genre en genre_kopie komen overeen.")
else:
    display(genre_verschillen)

Alle ingevulde waarden in genre en genre_kopie komen overeen.


In [4]:
#Alle overige kolommen verwijderen die niet relevant zijn voor de analyse
df = df.drop(columns=
             ['Unnamed: 3', 'materiaal_id', 'Themas_boek', 'woordenaantal_omschrijving', 
              'gem_woordlengte', 'gem_zinslengte_woorden', 'aandeel_lange_woorden_pct', 
              'aantal_zinnen', 'interne_batchcode', 'administratieve_prioriteit', 
              'genre_kopie', 'Gemiddelde beoordeling'])

In [5]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (283, 21)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,string,6
Auteur,231,7,string,45
Korte omschrijving,282,0,string,1
Type materiaal,10,1,string,272
Niveau,4,6,string,273
genre,3,26,string,254
onderwerp,47,181,string,55
trefwoorden,93,181,string,9
taal,4,0,string,279
aantal_paginas,159,17,Int64,107


In [6]:
df["aantal_paginas"] = df["aantal_paginas"].mask(
    df["aantal_paginas"] <= 0,
    np.nan
)

In [7]:
df["taal"] = (
    df["taal"]
    .str.strip()
    .str.lower()
    .replace({
        "ned": "Nederlands",
        "nl": "Nederlands",
        "nederlands": "Nederlands",
        "dutch": "Nederlands"
    })
)

In [8]:
df["Auteur"] = (
    df["Auteur"]
    .str.replace(r"\s*\(redactie\)", "", regex=True)
    .str.strip()
)

df.iloc[139:143]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,...,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,doelgroep_extra
139,Vanaf 2 augustus wordt duidelijker of het AI i...,Autoriteit Persoonsgegevens,"AI, privacy, deepfakes",online artikel,3F,informatief,<NA>,<NA>,Nederlands,2,...,https://www.autoriteitpersoonsgegevens.nl/actu...,MBO,Aangevuld (deels gesimuleerd),33.3,verhalen,Leeswereld,<NA>,<NA>,CAT-00140,MBO
140,Transparantie-eisen voor AI: wat betekent dit ...,Autoriteit Persoonsgegevens,"AI-verordening, rechten van burgers",online artikel,3F+,informatief,<NA>,<NA>,Nederlands,6,...,https://www.autoriteitpersoonsgegevens.nl/them...,MBO,Aangevuld (deels gesimuleerd),30.7,verhalen,<NA>,<NA>,online,CAT-00141,MBO
141,AI-verordening,Autoriteit Persoonsgegevens,Europese AI-wetgeving,online artikel,3F+,informatief,<NA>,<NA>,Nederlands,6,...,https://www.autoriteitpersoonsgegevens.nl/them...,MBO,gecontroleerd,49.0,verhalen,Educatief Media,<NA>,online,CAT-00142,MBO
142,"AP: AI-Impactbarometer kleurt rood, actie is n...",Autoriteit Persoonsgegevens,"AI, toezicht, discriminatie, recht",online artikel,3F+,<NA>,<NA>,<NA>,Nederlands,4,...,https://www.autoriteitpersoonsgegevens.nl/actu...,MBO,Aangevuld (deels gesimuleerd),44.7,verhalen,Educatief Media,2021,<NA>,CAT-00143,MBO/NT2


In [9]:
df["interessecluster"] = (
    df["interessecluster"]
    .str.strip()
    .str.lower()
    .str.capitalize()
)
df["interessecluster"].value_counts()

interessecluster
Verhalen        268
Maatschappij     13
Vrije tijd        2
Name: count, dtype: Int64

In [10]:
def merge_doelgroep(row):
    doelgroepen = set()

    for kolom in ["doelgroep", "doelgroep_extra"]:
        waarde = row.get(kolom)

        if pd.notna(waarde):
            for doelgroep in str(waarde).split("/"):
                doelgroep = doelgroep.strip().upper()

                if doelgroep:
                    doelgroepen.add(doelgroep)

    return " / ".join(sorted(doelgroepen))


df["doelgroep"] = df.apply(merge_doelgroep, axis=1)

df = df.drop(columns=["doelgroep_extra"])

In [11]:
df["doelgroep"] = df["doelgroep"].replace("", np.nan)

In [12]:
df["doelgroep"].value_counts()

doelgroep
MBO / NT2    212
MBO           65
NT2            3
Name: count, dtype: int64

In [13]:
df["Type materiaal"].value_counts()

Type materiaal
Boek               140
Boek - thriller     37
online artikel      34
Boek - roman        28
dichtbundel         23
blogpost            15
tijdschrift          2
boek                 1
krant (papier)       1
digitale krant       1
Name: count, dtype: Int64

In [14]:
def kapitaliseer_genres(genre):
    if pd.isna(genre):
        return genre

    return " / ".join(
        deel.strip().capitalize()
        for deel in str(genre).split("/")
    )

def verwerk_type_materiaal(df):
    # Bestaande genres eerst normaliseren
    df["genre"] = df["genre"].apply(kapitaliseer_genres)
    
    masker = (
        df["Type materiaal"]
        .fillna("")
        .str.strip()
        .str.lower()
        .str.startswith("boek -")
    )

    for index in df.index[masker]:
        extra_genre = df.at[index, "Type materiaal"].split("-", 1)[1].strip().capitalize()
        bestaand_genre = df.at[index, "genre"]

        if pd.isna(bestaand_genre) or str(bestaand_genre).strip() == "":
            df.at[index, "genre"] = extra_genre
        elif extra_genre.lower() not in str(bestaand_genre).lower():
            df.at[index, "genre"] = f"{bestaand_genre} / {extra_genre}"

    # Maak bijvoorbeeld "Boek - thriller" gelijk aan "Boek"
    df.loc[masker, "Type materiaal"] = "Boek"

    # Type materiaal normaliseren
    df["Type materiaal"] = (
        df["Type materiaal"]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({
            "digitale krant": "krant",
            "krant (papier)": "krant"
        })
    )

    # Boek met of zonder genre wordt Boek
    df.loc[
        df["Type materiaal"].str.match(r"^boek(?:\s*-.*)?$", na=False),
        "Type materiaal"
    ] = "boek"

    # Alleen de eerste letter hoofdletter maken
    df["Type materiaal"] = df["Type materiaal"].str.capitalize()
        
    return df

df = verwerk_type_materiaal(df)
df["Type materiaal"].value_counts()

Type materiaal
Boek              206
Online artikel     34
Dichtbundel        23
Blogpost           15
Tijdschrift         2
Krant               2
Name: count, dtype: Int64

In [15]:
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].str.capitalize().replace("Digitaal", "Online")

In [16]:
df["gegevensstatus"].value_counts()

gegevensstatus
Aangevuld (deels gesimuleerd)    245
gecontroleerd                     19
In behandeling                    11
Name: count, dtype: Int64

In [17]:
df["gegevensstatus"] = df["gegevensstatus"].replace({
    "Aangevuld (deels gesimuleerd)": "Beschikbaar",
    "gecontroleerd": "Niet beschikbaar",
    }).str.capitalize()

df["gegevensstatus"].value_counts()

gegevensstatus
Beschikbaar         245
Niet beschikbaar     19
In behandeling       11
Name: count, dtype: Int64

In [18]:
from datetime import datetime

huidig_jaar = datetime.now().year

df["publicatiejaar"] = pd.to_numeric(
    df["publicatiejaar"].replace("onbekend", pd.NA),
    errors="coerce"
)

df.loc[df["publicatiejaar"] > huidig_jaar, "publicatiejaar"] = pd.NA

df["publicatiejaar"] = df["publicatiejaar"].astype("Int64")

In [19]:
df["bron_url"] = (
    df["bron_url"]
    .str.replace(r"\s*\[.*$", "", regex=True)
    .str.strip()
)

In [20]:
df["onderwerp"] = df["onderwerp"].replace(
    r"^\[.*\]$",
    np.nan,
    regex=True
)

In [21]:
df["trefwoorden"] = df["trefwoorden"].replace(
    r"^\[.*\]$",
    np.nan,
    regex=True
)

In [22]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (283, 20)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,string,6
Auteur,230,7,string,46
Korte omschrijving,282,0,string,1
Type materiaal,6,1,string,276
Niveau,4,6,string,273
genre,7,20,str,256
onderwerp,46,183,string,54
trefwoorden,92,183,string,8
taal,1,0,string,282
aantal_paginas,157,28,Int64,98


In [23]:
#Missing values in de "Auteur" kolom vullen met "Onbekend"
df["Auteur"] = df["Auteur"].fillna("Onbekend")
df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,Boek,3F,Fictie,WOII,WOII; onderduik,Nederlands,265,556,<NA>,MBO,Beschikbaar,<NA>,Verhalen,Educatief Media,2025,Pdf,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,Fictie,spanning,spanning; stalking,Nederlands,171,184,<NA>,MBO / NT2,Beschikbaar,48.8,Verhalen,Leeswereld,2005,Online,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,Fictie,liefde,liefde; relaties,Nederlands,219,662,<NA>,MBO / NT2,Beschikbaar,49.2,Verhalen,<NA>,2025,Online,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,Fictie,ontmoeting,ontmoeting; verbondenheid,Nederlands,288,748,<NA>,MBO / NT2,Beschikbaar,55.3,Verhalen,Uitgeverij Noord,2018,Online,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,Fictie,geheimen,geheimen; schuld,Nederlands,358,196,<NA>,MBO / NT2,Beschikbaar,49.2,Verhalen,Uitgeverij Noord,<NA>,Online,CAT-00005


In [24]:
lege_rijen = df[
    df["Type materiaal"].isna() |
    df["Type materiaal"].astype(str).str.strip().eq("")
]

lege_rijen.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
67,Het geluid van de stilte,Inez Weski,"Persoonlijk relaas over rechtspraak, media-aan...",<NA>,3F+,NaN,<NA>,<NA>,Nederlands,200,617,<NA>,MBO,<NA>,54.0,Verhalen,Leeswereld,<NA>,Papier,CAT-00068


In [25]:
#in de "type materiaal" mist er 1 waarde waarvan we op hebben gezocht wat het is en dat invullen
df["Type materiaal"] = df["Type materiaal"].fillna("Boek")
df.iloc[67:68]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
67,Het geluid van de stilte,Inez Weski,"Persoonlijk relaas over rechtspraak, media-aan...",Boek,3F+,NaN,<NA>,<NA>,Nederlands,200,617,<NA>,MBO,<NA>,54.0,Verhalen,Leeswereld,<NA>,Papier,CAT-00068


In [26]:
lege_niveaus = df[
    df["Niveau"].isna() |
    df["Niveau"].astype(str).str.strip().eq("")
]

lege_niveaus.head(10)

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
37,Libello voor mannen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,Tijdschrift,<NA>,Fictie,<NA>,<NA>,Nederlands,354,837,<NA>,NaN,Niet beschikbaar,66.5,Verhalen,Leeswereld,<NA>,Papier,CAT-00038
38,Libello voor vrouwen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,Tijdschrift,<NA>,Fictie,<NA>,<NA>,Nederlands,153,248,<NA>,NT2,Beschikbaar,63.5,Verhalen,Uitgeverij Noord,2013,Papier,CAT-00039
39,De Startkrant,Onbekend,Eenvoudig geschreven krant met actueel nieuws ...,Krant,<NA>,Fictie,<NA>,<NA>,Nederlands,311,100,<NA>,NaN,Beschikbaar,60.0,Verhalen,Leeswereld,<NA>,Papier,CAT-00040
40,De Startweekkrant,Onbekend,Wekelijkse digitale krant met begrijpelijke ar...,Krant,<NA>,Fictie,<NA>,<NA>,Nederlands,244,386,<NA>,NT2,Beschikbaar,70.3,Verhalen,Leeswereld,<NA>,Online,CAT-00041
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",Online artikel,<NA>,Informatief,<NA>,<NA>,Nederlands,2,7,https://nos.nl/artikel/2548636-tekorten-zorgpe...,NaN,Beschikbaar,43.5,Verhalen,<NA>,2013,Pdf,CAT-00157
157,Nieuwe generatie fysiotherapeuten valt uit na ...,Sander Zurhake (NOS),"Fysiotherapie, zorg",Online artikel,<NA>,Informatief,<NA>,<NA>,Nederlands,1,14,https://nos.nl/artikel/2476626-nieuwe-generati...,NT2,Beschikbaar,44.6,Verhalen,Leeswereld,2023,Online,CAT-00158


In [27]:
#Missing values in de "Niveau" kolom vullen met "Onbekend"
# Lege waarden in Niveau herkennen
lege_niveaus = (
    df["Niveau"].isna()
    | df["Niveau"].astype(str).str.strip().eq("")
)
# Alleen lege Niveau-waarden aanpassen bij genre fictie
voorwaarde = (
    lege_niveaus
    & df["genre"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("fictie")
)

df.loc[voorwaarde, "Niveau"] = "2F"
df.iloc[37:41]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
37,Libello voor mannen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,Tijdschrift,2F,Fictie,<NA>,<NA>,Nederlands,354,837,<NA>,NaN,Niet beschikbaar,66.5,Verhalen,Leeswereld,<NA>,Papier,CAT-00038
38,Libello voor vrouwen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,Tijdschrift,2F,Fictie,<NA>,<NA>,Nederlands,153,248,<NA>,NT2,Beschikbaar,63.5,Verhalen,Uitgeverij Noord,2013,Papier,CAT-00039
39,De Startkrant,Onbekend,Eenvoudig geschreven krant met actueel nieuws ...,Krant,2F,Fictie,<NA>,<NA>,Nederlands,311,100,<NA>,NaN,Beschikbaar,60.0,Verhalen,Leeswereld,<NA>,Papier,CAT-00040
40,De Startweekkrant,Onbekend,Wekelijkse digitale krant met begrijpelijke ar...,Krant,2F,Fictie,<NA>,<NA>,Nederlands,244,386,<NA>,NT2,Beschikbaar,70.3,Verhalen,Leeswereld,<NA>,Online,CAT-00041


In [28]:
#Missing values in de "Niveau" kolom vullen met "Onbekend"
# Lege waarden in Niveau herkennen
lege_niveaus = (
    df["Niveau"].isna()
    | df["Niveau"].astype(str).str.strip().eq("")
)
# Alleen lege Niveau-waarden aanpassen bij genre fictie
voorwaarde = (
    lege_niveaus
    & df["genre"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("informatief")
)

df.loc[voorwaarde, "Niveau"] = "3F"
df.iloc[156:158]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",Online artikel,3F,Informatief,<NA>,<NA>,Nederlands,2,7,https://nos.nl/artikel/2548636-tekorten-zorgpe...,NaN,Beschikbaar,43.5,Verhalen,<NA>,2013,Pdf,CAT-00157
157,Nieuwe generatie fysiotherapeuten valt uit na ...,Sander Zurhake (NOS),"Fysiotherapie, zorg",Online artikel,3F,Informatief,<NA>,<NA>,Nederlands,1,14,https://nos.nl/artikel/2476626-nieuwe-generati...,NT2,Beschikbaar,44.6,Verhalen,Leeswereld,2023,Online,CAT-00158


In [29]:
df["aantal_paginas"].head()

0    265
1    171
2    219
3    288
4    358
Name: aantal_paginas, dtype: Int64

In [30]:
#Aantal paginas berekenen op basis van geschatte leestijd
#df["aantal_paginas"] = np.ceil(df["geschatte_leestijd_min"] / 2).astype("Int64")

#Alleen missing values in de "aantal_paginas" kolom vullen op basis van de geschatte leestijd
mask = df["aantal_paginas"].isna()

df.loc[mask, "aantal_paginas"] = np.ceil(
    df.loc[mask, "geschatte_leestijd_min"] / 2
).astype("Int64")
df["aantal_paginas"].head()

0    265
1    171
2    219
3    288
4    358
Name: aantal_paginas, dtype: Int64

In [31]:
#Geschatte leestijd berekenen op basis van geschatte leestijd
#df["geschatte_leestijd_min"] = np.ceil(df["aantal_paginas"] * 2).astype("Int64")

#Alleen missing values in de "geschatte_leestijd_min" kolom vullen op basis van de geschatte leestijd
mask = df["geschatte_leestijd_min"].isna()

df.loc[mask, "geschatte_leestijd_min"] = np.ceil(
    df.loc[mask, "aantal_paginas"] * 2
).astype("Int64")
df["geschatte_leestijd_min"].head()

0    556
1    184
2    662
3    748
4    196
Name: geschatte_leestijd_min, dtype: Int64

In [32]:
lege_rijen = df[
    df["doelgroep"].isna() |
    df["doelgroep"].astype(str).str.strip().eq("")
]

lege_rijen.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
37,Libello voor mannen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,Tijdschrift,2F,Fictie,<NA>,<NA>,Nederlands,354,837,<NA>,NaN,Niet beschikbaar,66.5,Verhalen,Leeswereld,<NA>,Papier,CAT-00038
39,De Startkrant,Onbekend,Eenvoudig geschreven krant met actueel nieuws ...,Krant,2F,Fictie,<NA>,<NA>,Nederlands,311,100,<NA>,NaN,Beschikbaar,60.0,Verhalen,Leeswereld,<NA>,Papier,CAT-00040
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",Online artikel,3F,Informatief,<NA>,<NA>,Nederlands,2,7,https://nos.nl/artikel/2548636-tekorten-zorgpe...,NaN,Beschikbaar,43.5,Verhalen,<NA>,2013,Pdf,CAT-00157


In [33]:
#Missing values in de "doelgroep" kolom vullen met "Onbekend"
#df["doelgroep"] = df["doelgroep"].fillna("Onbekend")

In [34]:
#Voeg extra kolom toe die aangeeft of de leescomplexiteit_score ontbreekt
df["leescomplexiteit_score_ontbreekt"] = (
    df["leescomplexiteit_score"].isna().astype(bool)
)

df["leescomplexiteit_score"] = df["leescomplexiteit_score"].fillna(df["leescomplexiteit_score"].median())

df[["Titel", "leescomplexiteit_score", "leescomplexiteit_score_ontbreekt"]].head()

,Titel,leescomplexiteit_score,leescomplexiteit_score_ontbreekt
0,Het dagboek van Anne Frank,45.4,True
1,Blauw Water,48.8,False
2,Boer vindt vrouw,49.2,False
3,Cadeau voor u!,55.3,False
4,De biecht,49.2,False


In [35]:
df["uitgever"] = df["uitgever"].fillna("Onbekend")

In [36]:
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].fillna("Onbekend")

In [37]:
df["gegevensstatus"] = df["gegevensstatus"].fillna("Onbekend")
df["gegevensstatus"].value_counts()

gegevensstatus
Beschikbaar         245
Niet beschikbaar     19
In behandeling       11
Onbekend              8
Name: count, dtype: Int64

In [38]:
df["bron_url"].isna().sum()

np.int64(234)

In [39]:
dubbele_titels = df[df["Titel"].duplicated(keep=False)]

for titel, groep in dubbele_titels.groupby("Titel"):
    verschillen = [
        kolom
        for kolom in df.columns
        if kolom != "Titel"
        and groep[kolom].nunique(dropna=False) > 1
    ]

    print(f"\nTitel: {titel}")
    print(f"Rijen: {list(groep.index)}")

    if verschillen:
        print("Verschillen gevonden in:", verschillen)
        display(groep[["Titel"] + verschillen])
    else:
        print("Geen verschillen gevonden.")


Titel: Gaat AI een rol spelen in de rechtspraak?
Rijen: [147, 177]
Verschillen gevonden in: ['Auteur', 'Korte omschrijving', 'Type materiaal', 'aantal_paginas', 'geschatte_leestijd_min', 'doelgroep', 'leescomplexiteit_score', 'uitgever', 'publicatiejaar', 'cataloguscode']


,Titel,Auteur,Korte omschrijving,Type materiaal,aantal_paginas,geschatte_leestijd_min,doelgroep,leescomplexiteit_score,uitgever,publicatiejaar,cataloguscode
147,Gaat AI een rol spelen in de rechtspraak?,KIJK-redactie,"Rechtspraak, AI, juridische innovatie",Online artikel,9,18,MBO / NT2,58.1,Uitgeverij Noord,2023,CAT-00148
177,Gaat AI een rol spelen in de rechtspraak?,KIJK Magazine,"Rechtspraak, technologie, AI",Blogpost,2,20,MBO,52.7,Onbekend,2013,CAT-00178



Titel: Nieuwe stap bij vernieuwing van curriculum
Rijen: [152, 182]
Verschillen gevonden in: ['Type materiaal', 'aantal_paginas', 'geschatte_leestijd_min', 'gegevensstatus', 'cataloguscode']


,Titel,Type materiaal,aantal_paginas,geschatte_leestijd_min,gegevensstatus,cataloguscode
152,Nieuwe stap bij vernieuwing van curriculum,Online artikel,1,8,Beschikbaar,CAT-00153
182,Nieuwe stap bij vernieuwing van curriculum,Blogpost,3,5,Onbekend,CAT-00183



Titel: Transparantie-eisen voor AI: wat betekent dit voor u?
Rijen: [140, 176]
Verschillen gevonden in: ['Korte omschrijving', 'Type materiaal', 'aantal_paginas', 'doelgroep', 'leescomplexiteit_score', 'uitgever', 'publicatiejaar', 'beschikbaar_formaat', 'cataloguscode']


,Titel,Korte omschrijving,Type materiaal,aantal_paginas,doelgroep,leescomplexiteit_score,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
140,Transparantie-eisen voor AI: wat betekent dit ...,"AI-verordening, rechten van burgers",Online artikel,6,MBO,30.7,Onbekend,<NA>,Online,CAT-00141
176,Transparantie-eisen voor AI: wat betekent dit ...,"AI-wetgeving, rechten van burgers",Blogpost,7,MBO / NT2,29.8,Leeswereld,2013,Pdf,CAT-00177



Titel: Vanaf 2 augustus wordt duidelijker of het AI is of echt
Rijen: [139, 174]
Verschillen gevonden in: ['Korte omschrijving', 'Type materiaal', 'aantal_paginas', 'geschatte_leestijd_min', 'doelgroep', 'leescomplexiteit_score', 'uitgever', 'publicatiejaar', 'beschikbaar_formaat', 'cataloguscode']


,Titel,Korte omschrijving,Type materiaal,aantal_paginas,geschatte_leestijd_min,doelgroep,leescomplexiteit_score,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
139,Vanaf 2 augustus wordt duidelijker of het AI i...,"AI, privacy, deepfakes",Online artikel,2,8,MBO,33.3,Leeswereld,<NA>,Onbekend,CAT-00140
174,Vanaf 2 augustus wordt duidelijker of het AI i...,"AI, deepfakes, consumentenrecht",Blogpost,1,13,MBO / NT2,54.9,Educatief Media,2018,Online,CAT-00175



Titel: Wat is een rechtsstaat?
Rijen: [143, 175, 181]
Verschillen gevonden in: ['Korte omschrijving', 'Type materiaal', 'aantal_paginas', 'geschatte_leestijd_min', 'leescomplexiteit_score', 'uitgever', 'publicatiejaar', 'cataloguscode']


,Titel,Korte omschrijving,Type materiaal,aantal_paginas,geschatte_leestijd_min,leescomplexiteit_score,uitgever,publicatiejaar,cataloguscode
143,Wat is een rechtsstaat?,"Rechtsstaat, grondrechten, democratie",Online artikel,8,6,74.3,Onbekend,<NA>,CAT-00144
175,Wat is een rechtsstaat?,"Rechtsstaat, democratie, burgerschap",Blogpost,7,11,73.6,Educatief Media,2005,CAT-00176
181,Wat is een rechtsstaat?,"Burgerschap, Nederland",Blogpost,2,5,70.4,Leeswereld,2018,CAT-00182


In [40]:
te_verwijderen = [139, 140, 143, 175, 177, 182]

# Rijen verwijderen
df = df.drop(index=te_verwijderen).copy()

# Index opnieuw nummeren
df = df.reset_index(drop=True)

# Cataloguscodes opnieuw en zonder gaten maken
df["cataloguscode"] = [
    f"CAT-{nummer:05d}"
    for nummer in range(1, len(df) + 1)
]

print(f"{len(te_verwijderen)} rij(en) verwijderd.")
print(f"{len(df)} rijen over.")

6 rij(en) verwijderd.
277 rijen over.


In [41]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (277, 21)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,string,0
Auteur,230,0,string,47
Korte omschrijving,277,0,string,0
Type materiaal,6,0,string,271
Niveau,4,0,string,273
genre,7,20,str,250
onderwerp,46,177,string,54
trefwoorden,92,177,string,8
taal,1,0,string,276
aantal_paginas,174,0,Int64,103


In [42]:
df = df.reset_index(drop=True)
df = df.convert_dtypes()
df.dtypes

Titel                                string
Auteur                               string
Korte omschrijving                   string
Type materiaal                       string
Niveau                               string
genre                                string
onderwerp                            string
trefwoorden                          string
taal                                 string
aantal_paginas                        Int64
geschatte_leestijd_min                Int64
bron_url                             string
doelgroep                            string
gegevensstatus                       string
leescomplexiteit_score              Float64
interessecluster                     string
uitgever                             string
publicatiejaar                        Int64
beschikbaar_formaat                  string
cataloguscode                        string
leescomplexiteit_score_ontbreekt    boolean
dtype: object

In [43]:
def maak_zoektekst(df):
    kolommen = [
        "Korte omschrijving", "Type materiaal", "Niveau", "genre",
        "onderwerp", "trefwoorden", "taal", "doelgroep"
    ]

    zoektekst = df[kolommen].apply(
        lambda rij: " ".join(
            rij.dropna().astype(str).str.strip()
        ),
        axis=1
    )

    return zoektekst


df["tekst_gecombineerd"] = maak_zoektekst(df)

df["tekst_gecombineerd"].iloc[[0]]

0    Dagboek van een Joods meisje dat tijdens de Tw...
Name: tekst_gecombineerd, dtype: str

In [44]:
import spacy

nlp = spacy.load("nl_core_news_lg")

In [45]:
df["tekst_lowercase"] = (
    df["tekst_gecombineerd"]
    .str.lower()
    .str.strip()
)

df["tekst_lowercase"].iloc[[0]]

0    dagboek van een joods meisje dat tijdens de tw...
Name: tekst_lowercase, dtype: str

In [46]:
def tekst_opschonen(tekst):
    # Verwijder losse cijfers, maar behoud bijvoorbeeld 2f en 3f
    tekst = re.sub(r"(?<![A-Za-zÀ-ÿ])\d+(?![A-Za-zÀ-ÿ])", " ", tekst)

    # Verwijder leestekens, maar behoud letters, cijfers en spaties
    tekst = re.sub(r"[^A-Za-zÀ-ÿ0-9\s]", " ", tekst)

    # Maak meerdere spaties één spatie
    tekst = re.sub(r"\s+", " ", tekst)

    return tekst.strip()

df["tekst_schoon"] = df["tekst_lowercase"].apply(tekst_opschonen)

df["tekst_schoon"].iloc[[0]]

0    dagboek van een joods meisje dat tijdens de tw...
Name: tekst_schoon, dtype: str

In [47]:
df["tokens"] = df["tekst_schoon"].apply(
    lambda tekst: [token.text for token in nlp(tekst)]
)

df["tokens"].iloc[[0]]

0    [dagboek, van, een, joods, meisje, dat, tijden...
Name: tokens, dtype: object

In [48]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

stopwoorden = set(stopwords.words("dutch"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\addov\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [49]:
def verwijder_stopwoorden(tekst):
    tokens = tekst.split()

    return [
        token
        for token in tokens
        if token not in stopwoorden
    ]

df["tokens_zonder_stopwoorden"] = df["tekst_schoon"].apply(
    verwijder_stopwoorden
)

df["tokens_zonder_stopwoorden"].iloc[[0]]

0    [dagboek, joods, meisje, tijdens, tweede, were...
Name: tokens_zonder_stopwoorden, dtype: object

In [50]:
def lemmatiseer(tekst):
    doc = nlp(tekst)

    resultaat = []

    for token in doc:
        is_niveau = re.fullmatch(r"\d+[a-zA-Z]", token.text)

        if token.text.lower() in stopwoorden:
            continue

        if token.is_alpha:
            resultaat.append(token.lemma_)
        elif is_niveau:
            resultaat.append(token.text.upper())

    return resultaat

df["tokens_basisvorm"] = df["tekst_schoon"].apply(lemmatiseer)

df["tokens_basisvorm"].iloc[[0]]

0    [dagboek, joods, meisje, tijdens, twee, wereld...
Name: tokens_basisvorm, dtype: object

In [51]:
df["tekst_nlp"] = df["tokens_basisvorm"].apply(
    lambda tokens: " ".join(tokens)
)

df["tekst_nlp"].iloc[0:3]

0    dagboek joods meisje tijdens twee wereldoorlog...
1    spannend thriller waarin vrouw confronteren st...
2    toegankelijk verhaal liefde relatie zoektocht ...
Name: tekst_nlp, dtype: str

In [52]:
df.iloc[0:3]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,...,beschikbaar_formaat,cataloguscode,leescomplexiteit_score_ontbreekt,tekst_gecombineerd,tekst_lowercase,tekst_schoon,tokens,tokens_zonder_stopwoorden,tokens_basisvorm,tekst_nlp
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,Boek,3F,Fictie,WOII,WOII; onderduik,Nederlands,265,...,Pdf,CAT-00001,True,Dagboek van een Joods meisje dat tijdens de Tw...,dagboek van een joods meisje dat tijdens de tw...,dagboek van een joods meisje dat tijdens de tw...,"[dagboek, van, een, joods, meisje, dat, tijden...","[dagboek, joods, meisje, tijdens, tweede, were...","[dagboek, joods, meisje, tijdens, twee, wereld...",dagboek joods meisje tijdens twee wereldoorlog...
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,Fictie,spanning,spanning; stalking,Nederlands,171,...,Online,CAT-00002,False,Spannende thriller waarin een vrouw wordt geco...,spannende thriller waarin een vrouw wordt geco...,spannende thriller waarin een vrouw wordt geco...,"[spannende, thriller, waarin, een, vrouw, word...","[spannende, thriller, waarin, vrouw, geconfron...","[spannend, thriller, waarin, vrouw, confronter...",spannend thriller waarin vrouw confronteren st...
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,Fictie,liefde,liefde; relaties,Nederlands,219,...,Online,CAT-00003,False,"Toegankelijk verhaal over liefde, relaties en ...","toegankelijk verhaal over liefde, relaties en ...",toegankelijk verhaal over liefde relaties en d...,"[toegankelijk, verhaal, over, liefde, relaties...","[toegankelijk, verhaal, liefde, relaties, zoek...","[toegankelijk, verhaal, liefde, relatie, zoekt...",toegankelijk verhaal liefde relatie zoektocht ...


In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b"
)

tfidf_matrix = vectorizer.fit_transform(df["tekst_nlp"])

tfidf_matrix.shape

(277, 753)

In [54]:
woorden = vectorizer.get_feature_names_out()

woorden[:20]

array(['2F', '3F', 'Amsterdam', 'Anne', 'Bedrog', 'Biodiversiteit',
       'Dichtbundel', 'Duits', 'Eerstejaars', 'Engeland', 'Europees',
       'Fictieboek', 'Holmes', 'Kansenongelijkheid', 'Maatschappij',
       'Migratie', 'Nederland', 'Nederlands', 'Rika', 'Sport'],
      dtype=object)

In [55]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=woorden,
    index=df.index
)

tfidf_df.head()

,2F,3F,Amsterdam,Anne,Bedrog,Biodiversiteit,Dichtbundel,Duits,Eerstejaars,Engeland,...,zitten,zoals,zoeken,zoektocht,zomers,zorg,zorgen,zwaar,zwangerschap,én
0,0.000000,0.062398,0.218703,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.071651,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,0.075205,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.190642,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.070643,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,0.080728,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [56]:
boek_1 = tfidf_df.iloc[0]

display(
    boek_1[boek_1 > 0]
    .sort_values(ascending=False)
)

woii            0.437407
onderduiken     0.407521
beeld           0.218703
dagboek         0.218703
wereldoorlog    0.218703
vervolging      0.218703
joods           0.218703
Amsterdam       0.218703
angst           0.203761
observatie      0.203761
geven           0.193159
dagelijks       0.178216
hopen           0.172535
groei           0.167614
tijdens         0.167614
indringen       0.167614
meisje          0.159391
twee            0.149722
leven           0.122786
persoonlijk     0.122786
3F              0.062398
fictie          0.050493
boek            0.047543
mbo             0.037657
Nederlands      0.036853
Name: 0, dtype: float64

In [57]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_matrix.shape

(277, 277)

In [58]:
vergelijkbare_boeken = similarity_matrix[0].argsort()[::-1][1:6]

In [59]:
resultaat = pd.DataFrame({
    "index": vergelijkbare_boeken,
    "similarity": similarity_matrix[0, vergelijkbare_boeken],
    "tekst": df.iloc[vergelijkbare_boeken]["tekst_nlp"].to_numpy()
})

display(resultaat)

,index,similarity,tekst
0,241,0.285738,novelle waarin onderduiken spanning centraal s...
1,162,0.150205,liefde werk dagelijks leven dichtbundel 3F poë...
2,84,0.149303,historisch roman jong meisje tijdens twee Were...
3,128,0.145738,indringen roman identiteit familie persoonlijk...
4,58,0.134329,romantisch verhaal liefde timing persoonlijk g...
